In [1]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-stack", download=True)
db = dataset.get_db()


/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/abed/.cache/relbench/rel-stack/db...
Done in 9.22 seconds.


In [2]:
task = get_task("rel-stack", "user-engagement", download=True)
train_table = task.get_table("train")
val_table   = task.get_table("val")
test_table  = task.get_table("test")

In [3]:
print("Tables in DB:", list(db.table_dict.keys()))
for name, tbl in db.table_dict.items():
    print(f"\n=== {name} ===  shape={tbl.df.shape}")
    print(tbl.df.dtypes)
    print(tbl.df.head(2))


Tables in DB: ['users', 'postHistory', 'postLinks', 'votes', 'posts', 'badges', 'comments']

=== users ===  shape=(255360, 8)
Id                          Int64
AccountId                 float64
DisplayName                object
Location                   object
ProfileImageUrl           float64
WebsiteUrl                 object
AboutMe                    object
CreationDate       datetime64[ns]
dtype: object
   Id  AccountId   DisplayName            Location  ProfileImageUrl  \
0   0       -1.0     Community  on the server farm              NaN   
1   1        2.0  Geoff Dalgas       Corvallis, OR              NaN   

                       WebsiteUrl  \
0  http://meta.stackexchange.com/   
1        http://stackoverflow.com   

                                             AboutMe            CreationDate  
0  <p>Hi, I'm not really a person.</p>\n\n<p>I'm ... 2010-07-19 06:55:26.860  
1  <p>Dev #2 who helped create Stack Overflow cur... 2010-07-19 14:01:36.697  

=== postHistory ===  sha

In [4]:
for name, t in [("train", train_table), ("val", val_table), ("test", test_table)]:
    df = t.df
    print(f"{name}: shape={df.shape}, cols={list(df.columns)}")
    print(df.head(2))
    if "contribution" in df.columns:
        print("  positive rate:", df["contribution"].mean())


train: shape=(1360850, 3), cols=['timestamp', 'OwnerUserId', 'contribution']
   timestamp  OwnerUserId  contribution
0 2012-01-12          352             1
1 2010-10-14            7             1
  positive rate: 0.04998346621596796
val: shape=(85838, 3), cols=['timestamp', 'OwnerUserId', 'contribution']
   timestamp  OwnerUserId  contribution
0 2020-10-01       246773             1
1 2020-10-01       199619             1
  positive rate: 0.02808779328502528
test: shape=(88137, 2), cols=['timestamp', 'OwnerUserId']
   timestamp  OwnerUserId
0 2021-01-01          668
1 2021-01-01       125849


In [5]:
for name, tbl in db.table_dict.items():
    print(f"{name}: pkey={tbl.pkey_col}, fkeys={tbl.fkey_col_to_pkey_table}")


users: pkey=Id, fkeys={}
postHistory: pkey=Id, fkeys={'PostId': 'posts', 'UserId': 'users'}
postLinks: pkey=Id, fkeys={'PostId': 'posts', 'RelatedPostId': 'posts'}
votes: pkey=Id, fkeys={'PostId': 'posts', 'UserId': 'users'}
posts: pkey=Id, fkeys={'OwnerUserId': 'users', 'ParentId': 'posts'}
badges: pkey=Id, fkeys={'UserId': 'users'}
comments: pkey=Id, fkeys={'UserId': 'users', 'PostId': 'posts'}


## Q1 — Three manual join-aggregate features

In [6]:
import pandas as pd
import numpy as np

users_df    = db.table_dict["users"].df
posts_df    = db.table_dict["posts"].df
votes_df    = db.table_dict["votes"].df
comments_df = db.table_dict["comments"].df

UID = "OwnerUserId"  # user-id column in the target table

def _count_before(target_df, src_df, user_col, name):
    """Count rows of src_df per (user, timestamp) where src CreationDate <= timestamp."""
    s = src_df[[user_col, "CreationDate"]].dropna(subset=[user_col]).copy()
    s[user_col] = s[user_col].astype("int64")
    s = s.rename(columns={user_col: UID})
    m = target_df[[UID, "timestamp"]].merge(s, on=UID, how="left")
    valid = m["CreationDate"].notna() & (m["CreationDate"] <= m["timestamp"])
    return (m.assign(valid=valid)
              .groupby([UID, "timestamp"])["valid"].sum()
              .astype("int64").rename(name))

def build_q1_features(target_df):
    cols = [UID, "timestamp"] + (["contribution"] if "contribution" in target_df.columns else [])
    out = target_df[cols].copy()

    # Feature 1: # posts owned by user before timestamp
    f1 = _count_before(out, posts_df, "OwnerUserId", "num_posts_before_t")

    # Feature 2: # votes cast by user before timestamp
    f2 = _count_before(out, votes_df, "UserId", "num_votes_before_t")

    # Feature 3: days since user's last comment (NaN if user never commented before t)
    c = comments_df[["UserId", "CreationDate"]].dropna(subset=["UserId"]).copy()
    c["UserId"] = c["UserId"].astype("int64")
    c = c.rename(columns={"UserId": UID})
    m = out[[UID, "timestamp"]].merge(c, on=UID, how="left")
    m = m[m["CreationDate"].notna() & (m["CreationDate"] <= m["timestamp"])]
    last_c = m.groupby([UID, "timestamp"])["CreationDate"].max().rename("last_comment_t")
    joined = out.set_index([UID, "timestamp"]).join(last_c)
    delta = joined.index.get_level_values("timestamp") - joined["last_comment_t"]
    f3 = (delta.dt.total_seconds() / 86400.0).rename("days_since_last_comment")

    out = out.set_index([UID, "timestamp"]).join([f1, f2, f3]).reset_index()
    return out

train_q1 = build_q1_features(train_table.df)
print("shape:", train_q1.shape)
print(train_q1.head())
print("\nNaN counts:\n", train_q1.isna().sum())
print("\nDescribe:\n", train_q1[["num_posts_before_t", "num_votes_before_t", "days_since_last_comment"]].describe())

shape: (1360850, 6)
   OwnerUserId  timestamp  contribution  num_posts_before_t  \
0          352 2012-01-12             1                 218   
1            7 2010-10-14             1                  80   
2         1693 2011-01-13             1                   6   
3         1160 2011-04-14             1                   7   
4         1895 2011-04-14             1                  31   

   num_votes_before_t  days_since_last_comment  
0                   2                 2.231586  
1                   0                 0.621124  
2                   0                 6.150768  
3                   0               157.357496  
4                   0                 0.272961  

NaN counts:
 OwnerUserId                     0
timestamp                       0
contribution                    0
num_posts_before_t              0
num_votes_before_t              0
days_since_last_comment    545289
dtype: int64

Describe:
        num_posts_before_t  num_votes_before_t  days_since_last_c

## Q2 — Automatic join-aggregate feature generation

In [7]:
from collections import defaultdict
import pandas as pd
import numpy as np

# ---------- schema graph ----------
def build_schema_graph(db):
    """Each edge: (neighbor_table, col_in_curr, col_in_neighbor, edge_label)."""
    schema = db.table_dict
    adj = defaultdict(list)
    for tname, tbl in schema.items():
        for fk_col, ref_table in tbl.fkey_col_to_pkey_table.items():
            ref_pk = schema[ref_table].pkey_col
            # forward edge: tname --(fk_col -> ref_pk)--> ref_table
            adj[tname].append((ref_table, fk_col, ref_pk, f"{tname}.{fk_col}->{ref_table}.{ref_pk}"))
            # reverse edge: ref_table --(ref_pk -> fk_col)--> tname
            adj[ref_table].append((tname, ref_pk, fk_col, f"{ref_table}.{ref_pk}->{tname}.{fk_col}"))
    return adj

def enumerate_paths(adj, start, max_depth):
    """All simple paths starting at `start` of length 1..max_depth.
    Each path is a list of edges (neighbor, col_in_prev, col_in_neighbor, label)."""
    out = []
    def dfs(curr, visited, edges):
        if len(edges) >= 1:
            out.append(list(edges))
        if len(edges) >= max_depth:
            return
        for (nxt, lcol, rcol, lbl) in adj[curr]:
            if nxt in visited: 
                continue
            visited.add(nxt)
            edges.append((nxt, lcol, rcol, lbl))
            dfs(nxt, visited, edges)
            edges.pop()
            visited.remove(nxt)
    dfs(start, {start}, [])
    return out

# ---------- per-table info: which column is the "creation time", and which cols are aggregable ----------
TIME_COL = {
    "users": "CreationDate", "posts": "CreationDate", "postHistory": "CreationDate",
    "postLinks": "CreationDate", "votes": "CreationDate", "comments": "CreationDate",
    "badges": "Date",
}

def aggregable_columns(db, table_name):
    """Return list of (col_name, kind) for columns we will aggregate.
    Literal PDF reading: aggregate every column in the joined table (no PK/FK skips)."""
    tbl = db.table_dict[table_name]
    out = []
    for c, dtype in tbl.df.dtypes.items():
        if pd.api.types.is_datetime64_any_dtype(dtype):
            out.append((c, "time"))
        elif pd.api.types.is_numeric_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
            out.append((c, "num"))
        else:
            out.append((c, "text"))
    return out

def count_features_for_depth(db, max_depth):
    adj = build_schema_graph(db)
    paths = enumerate_paths(adj, "users", max_depth - 1)  # depth-1 here = target⟕users (0 edges from users)
    # depth-1 (the assignment's "1 hop") corresponds to ZERO edges from `users` => just the users table itself
    n = 0
    # depth-1: the "users" path
    cols = aggregable_columns(db, "users")
    n += sum(3 if k == "num" else (1 if k == "text" else 2) for _, k in cols)
    # depth-d for d >= 2: paths of length d-1 from users
    for edges in paths:
        leaf = edges[-1][0]
        cols = aggregable_columns(db, leaf)
        n += sum(3 if k == "num" else (1 if k == "text" else 2) for _, k in cols)
    return n

# sanity check
adj = build_schema_graph(db)
print("schema adjacency:")
for k, vs in adj.items():
    print(" ", k, "->", [(v[0], v[3]) for v in vs])

print("\nQ3: feature counts by depth")
for d in [1, 2, 3]:
    print(f"  max_depth={d}: {count_features_for_depth(db, d)} features")

schema adjacency:
  postHistory -> [('posts', 'postHistory.PostId->posts.Id'), ('users', 'postHistory.UserId->users.Id')]
  posts -> [('postHistory', 'posts.Id->postHistory.PostId'), ('postLinks', 'posts.Id->postLinks.PostId'), ('postLinks', 'posts.Id->postLinks.RelatedPostId'), ('votes', 'posts.Id->votes.PostId'), ('users', 'posts.OwnerUserId->users.Id'), ('posts', 'posts.ParentId->posts.Id'), ('posts', 'posts.Id->posts.ParentId'), ('comments', 'posts.Id->comments.PostId')]
  users -> [('postHistory', 'users.Id->postHistory.UserId'), ('votes', 'users.Id->votes.UserId'), ('posts', 'users.Id->posts.OwnerUserId'), ('badges', 'users.Id->badges.UserId'), ('comments', 'users.Id->comments.UserId')]
  postLinks -> [('posts', 'postLinks.PostId->posts.Id'), ('posts', 'postLinks.RelatedPostId->posts.Id')]
  votes -> [('posts', 'votes.PostId->posts.Id'), ('users', 'votes.UserId->users.Id')]
  badges -> [('users', 'badges.UserId->users.Id')]
  comments -> [('users', 'comments.UserId->users.Id'), (

In [8]:
# ----- Q2: materialize the feature_matrix at max_depth=2 -----
import time

UID = "OwnerUserId"

def _agg_dict(cols_info):
    d = {}
    for c, kind in cols_info:
        if kind == "num":
            d[c] = ["mean", "count", "sum"]
        elif kind == "text":
            d[c] = ["count"]
        else:  # time
            d[c] = ["min", "max"]
    return d

def build_feature_matrix(target_df, db, max_depth=2, verbose=True):
    assert max_depth == 2, "this implementation is specialized to depth=2"
    t0 = time.time()
    base_cols = [UID, "timestamp"] + (["contribution"] if "contribution" in target_df.columns else [])
    fm = target_df[base_cols].reset_index(drop=True).copy()
    fm["_tid"] = np.arange(len(fm), dtype=np.int64)

    # ---------- depth-1: target ⟕ users ----------
    if verbose: print("[depth-1] target ⟕ users ...")
    users_df = db.table_dict["users"].df.copy()
    users_df["Id"] = users_df["Id"].astype("int64")
    j = fm[["_tid", UID]].merge(users_df, left_on=UID, right_on="Id", how="left").set_index("_tid")
    new_cols = {}
    for col, kind in aggregable_columns(db, "users"):
        if kind == "num":
            new_cols[f"users__{col}__mean"]  = j[col].astype("float64")
            new_cols[f"users__{col}__count"] = j[col].notna().astype("int64")
            new_cols[f"users__{col}__sum"]   = j[col].astype("float64")
        elif kind == "text":
            new_cols[f"users__{col}__count"] = j[col].notna().astype("int64")
        else:  # time
            new_cols[f"users__{col}__min"]   = j[col]
            new_cols[f"users__{col}__max"]   = j[col]
    d1 = pd.DataFrame(new_cols)

    # ---------- depth-2: target ⟕ users ⋈ X (for each user-neighbor X) ----------
    depth2 = [
        ("posts",       "OwnerUserId"),
        ("postHistory", "UserId"),
        ("votes",       "UserId"),
        ("badges",      "UserId"),
        ("comments",    "UserId"),
    ]
    depth2_frames = []
    for tname, user_fk in depth2:
        if verbose: print(f"[depth-2] users ⋈ {tname} (on {user_fk}) ...", end=" ")
        time_col = TIME_COL[tname]
        cols_info = aggregable_columns(db, tname)
        src = db.table_dict[tname].df.copy()
        src = src.dropna(subset=[user_fk])
        src[user_fk] = src[user_fk].astype("int64")

        m = fm[["_tid", UID, "timestamp"]].merge(src, left_on=UID, right_on=user_fk, how="inner")
        m = m[m[time_col] <= m["timestamp"]]
        if verbose: print(f"matched rows: {len(m):,}")
        if len(m) == 0:
            continue

        g = m.groupby("_tid").agg(_agg_dict(cols_info))
        g.columns = [f"{tname}__{c}__{a}" for c, a in g.columns]
        depth2_frames.append(g)

    # ---------- assemble ----------
    feature_matrix = (
        fm.set_index("_tid")
          .join(d1)
          .join(depth2_frames, how="left")
    )
    count_cols = [c for c in feature_matrix.columns if c.endswith("__count")]
    feature_matrix[count_cols] = feature_matrix[count_cols].fillna(0).astype("int64")
    feature_matrix = feature_matrix.reset_index(drop=True)
    if verbose: print(f"feature_matrix shape: {feature_matrix.shape}   ({time.time()-t0:.1f}s)")
    return feature_matrix

fm_train = build_feature_matrix(train_table.df, db, max_depth=2)
print("\ncolumns ({}):".format(len(fm_train.columns)))
print(list(fm_train.columns))
fm_train.head(2)


[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 5,250,955
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 16,814,778
[depth-2] users ⋈ votes (on UserId) ... matched rows: 77,552
[depth-2] users ⋈ badges (on UserId) ... matched rows: 5,425,886
[depth-2] users ⋈ comments (on UserId) ... matched rows: 9,705,158
feature_matrix shape: (1360850, 99)   (37.1s)

columns (99):
['OwnerUserId', 'timestamp', 'contribution', 'users__Id__mean', 'users__Id__count', 'users__Id__sum', 'users__AccountId__mean', 'users__AccountId__count', 'users__AccountId__sum', 'users__DisplayName__count', 'users__Location__count', 'users__ProfileImageUrl__mean', 'users__ProfileImageUrl__count', 'users__ProfileImageUrl__sum', 'users__WebsiteUrl__count', 'users__AboutMe__count', 'users__CreationDate__min', 'users__CreationDate__max', 'posts__Id__mean', 'posts__Id__count', 'posts__Id__sum', 'posts__OwnerUserId__mean', 'posts__OwnerUserId__count', 'posts__OwnerUserId__su

,OwnerUserId,timestamp,contribution,users__Id__mean,users__Id__count,users__Id__sum,users__AccountId__mean,users__AccountId__count,users__AccountId__sum,users__DisplayName__count,...,comments__PostId__count,comments__PostId__sum,comments__UserId__mean,comments__UserId__count,comments__UserId__sum,comments__ContentLicense__count,comments__UserDisplayName__count,comments__Text__count,comments__CreationDate__min,comments__CreationDate__max
0,352,2012-01-12,1,352.0,1,352.0,229230.0,1,229230.0,1,...,353,2399604,352.0,353,124256.0,353,0,353,2010-08-20 12:42:10.483,2012-01-09 18:26:30.950
1,7,2010-10-14,1,7.0,1,7.0,70002.0,1,70002.0,1,...,72,122775,7.0,72,504.0,72,0,72,2010-07-19 19:56:18.490,2010-10-13 09:05:34.927


In [9]:
# Build feature matrices for val/test too, and persist all three to disk.
import os
os.makedirs("features", exist_ok=True)

fm_val  = build_feature_matrix(val_table.df,  db, max_depth=2)
fm_test = build_feature_matrix(test_table.df, db, max_depth=2)

fm_train.to_pickle("features/fm_train.pkl")
fm_val.to_pickle("features/fm_val.pkl")
fm_test.to_pickle("features/fm_test.pkl")

print(f"\nsaved:")
print(f"  fm_train: {fm_train.shape}")
print(f"  fm_val:   {fm_val.shape}")
print(f"  fm_test:  {fm_test.shape}")

[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 319,820
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 1,066,932
[depth-2] users ⋈ votes (on UserId) ... matched rows: 5,017
[depth-2] users ⋈ badges (on UserId) ... matched rows: 367,072
[depth-2] users ⋈ comments (on UserId) ... matched rows: 594,598
feature_matrix shape: (85838, 99)   (3.3s)
[depth-1] target ⟕ users ...
[depth-2] users ⋈ posts (on OwnerUserId) ... matched rows: 328,648
[depth-2] users ⋈ postHistory (on UserId) ... matched rows: 1,098,857
[depth-2] users ⋈ votes (on UserId) ... matched rows: 5,182
[depth-2] users ⋈ badges (on UserId) ... matched rows: 380,489
[depth-2] users ⋈ comments (on UserId) ... matched rows: 612,288
feature_matrix shape: (88137, 98)   (2.4s)

saved:
  fm_train: (1360850, 99)
  fm_val:   (85838, 99)
  fm_test:  (88137, 98)


## Q3–Q6 — Feature counts, shape, examples, missing values

## Q7 — Train and compare four classifiers

In [10]:
# ---------- Preprocessing for Q7 ----------
import numpy as np
import pandas as pd

def prepare_xy(fm, drop_cols=("OwnerUserId", "timestamp", "contribution")):
    """Return (X DataFrame with numeric features only, y array or None)."""
    y = fm["contribution"].values.astype("int64") if "contribution" in fm.columns else None
    X = fm.drop(columns=[c for c in drop_cols if c in fm.columns]).copy()
    # convert datetime cols to int64 ns (NaN -> -1)
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            X[c] = X[c].astype("int64")  # NaT becomes a large negative; we'll normalize below
            X[c] = X[c].mask(X[c] < 0, -1)
        elif X[c].dtype == "object":
            # shouldn't happen after our aggregations, but guard
            X[c] = pd.to_numeric(X[c], errors="coerce")
        else:
            X[c] = X[c].astype("float64")
    return X, y

X_train, y_train = prepare_xy(fm_train)
X_val,   y_val   = prepare_xy(fm_val)

print("X_train:", X_train.shape, " y_train pos-rate:", y_train.mean().round(4))
print("X_val:  ", X_val.shape,   " y_val pos-rate:  ", y_val.mean().round(4))
print("dtypes:", X_train.dtypes.value_counts().to_dict())
print("nan share per col (top 5):")
print(X_train.isna().mean().sort_values(ascending=False).head(5))

X_train: (1360850, 96)  y_train pos-rate: 0.05
X_val:   (85838, 96)  y_val pos-rate:   0.0281
dtypes: {dtype('float64'): 84, dtype('int64'): 12}
nan share per col (top 5):
users__ProfileImageUrl__sum     1.000000
users__ProfileImageUrl__mean    1.000000
votes__PostId__mean             0.970992
votes__UserId__mean             0.969956
votes__VoteTypeId__mean         0.969956
dtype: float64


In [12]:
# ---------- Eval harness ----------
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
)

def eval_split(y_true, y_pred, y_score):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def fmt(metrics_tr, metrics_val):
    return {k: f"{metrics_tr[k]:.3f}/{metrics_val[k]:.3f}" for k in metrics_tr}

# Big dict to collect every variant we try, for the per-model variant tables in the report.
all_runs = []
def log_run(model_name, variant, m_tr, m_val):
    all_runs.append({"model": model_name, "variant": variant,
                     **{f"tr_{k}": v for k, v in m_tr.items()},
                     **{f"val_{k}": v for k, v in m_val.items()}})

In [13]:
# ---------- Decision Tree ----------
from sklearn.tree import DecisionTreeClassifier

# DT in sklearn < 1.3 does NOT handle NaN. Fill with -1 sentinel.
X_train_dt = X_train.fillna(-1).values
X_val_dt   = X_val.fillna(-1).values

dt_grid = [
    {"max_depth": 5},
    {"max_depth": 10},
    {"max_depth": 20},
    {"max_depth": None, "min_samples_leaf": 50},
]
best_dt, best_dt_val_auc = None, -1
for params in dt_grid:
    clf = DecisionTreeClassifier(class_weight="balanced", random_state=0, **params)
    clf.fit(X_train_dt, y_train)
    s_tr  = clf.predict_proba(X_train_dt)[:, 1]
    s_val = clf.predict_proba(X_val_dt)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("DecisionTree", str(params), m_tr, m_val)
    print(f"DT {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_dt_val_auc:
        best_dt, best_dt_val_auc = clf, m_val["roc_auc"]
        best_dt_params, best_dt_m_tr, best_dt_m_val = params, m_tr, m_val

print("\nBEST DT:", best_dt_params, "val AUC =", round(best_dt_val_auc, 3))

DT {'max_depth': 5}: val AUC=0.862, val F1=0.144
DT {'max_depth': 10}: val AUC=0.858, val F1=0.146
DT {'max_depth': 20}: val AUC=0.663, val F1=0.186
DT {'max_depth': None, 'min_samples_leaf': 50}: val AUC=0.841, val F1=0.158

BEST DT: {'max_depth': 5} val AUC = 0.862


In [14]:
# ---------- XGBoost ----------
from xgboost import XGBClassifier

pos_ratio = float((y_train == 0).sum() / (y_train == 1).sum())  # ~19 for ~5% positive

xgb_grid = [
    {"max_depth": 4,  "n_estimators": 200, "learning_rate": 0.10},
    {"max_depth": 6,  "n_estimators": 200, "learning_rate": 0.10},
    {"max_depth": 8,  "n_estimators": 400, "learning_rate": 0.05},
]
best_xgb, best_xgb_val_auc = None, -1
for params in xgb_grid:
    clf = XGBClassifier(
        scale_pos_weight=pos_ratio,
        tree_method="hist", n_jobs=-1, random_state=0,
        eval_metric="logloss", **params,
    )
    clf.fit(X_train.values, y_train)
    s_tr  = clf.predict_proba(X_train.values)[:, 1]
    s_val = clf.predict_proba(X_val.values)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("XGBoost", str(params), m_tr, m_val)
    print(f"XGB {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_xgb_val_auc:
        best_xgb, best_xgb_val_auc = clf, m_val["roc_auc"]
        best_xgb_params, best_xgb_m_tr, best_xgb_m_val = params, m_tr, m_val

print("\nBEST XGB:", best_xgb_params, "val AUC =", round(best_xgb_val_auc, 3))

XGB {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.882, val F1=0.152
XGB {'max_depth': 6, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.879, val F1=0.159
XGB {'max_depth': 8, 'n_estimators': 400, 'learning_rate': 0.05}: val AUC=0.874, val F1=0.173

BEST XGB: {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1} val AUC = 0.882


In [15]:
# ---------- Logistic Regression ----------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median").fit(X_train.values)
X_tr_imp = imp.transform(X_train.values)
X_va_imp = imp.transform(X_val.values)
scaler = StandardScaler().fit(X_tr_imp)
X_tr_sc = scaler.transform(X_tr_imp)
X_va_sc = scaler.transform(X_va_imp)

lr_grid = [
    {"C": 0.1},
    {"C": 1.0},
    {"C": 10.0},
]
best_lr, best_lr_val_auc = None, -1
for params in lr_grid:
    clf = LogisticRegression(
        class_weight="balanced", max_iter=2000, solver="lbfgs",
        n_jobs=-1, random_state=0, **params,
    )
    clf.fit(X_tr_sc, y_train)
    s_tr  = clf.predict_proba(X_tr_sc)[:, 1]
    s_val = clf.predict_proba(X_va_sc)[:, 1]
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("LogReg", str(params), m_tr, m_val)
    print(f"LR {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_lr_val_auc:
        best_lr, best_lr_val_auc = clf, m_val["roc_auc"]
        best_lr_params, best_lr_m_tr, best_lr_m_val = params, m_tr, m_val

print("\nBEST LR:", best_lr_params, "val AUC =", round(best_lr_val_auc, 3))

/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 8 10]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 8 10]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


LR {'C': 0.1}: val AUC=0.850, val F1=0.157
LR {'C': 1.0}: val AUC=0.849, val F1=0.157
LR {'C': 10.0}: val AUC=0.849, val F1=0.156

BEST LR: {'C': 0.1} val AUC = 0.85


### Custom NN (MLP) — architecture

In [16]:
# ---------- Custom NN (PyTorch MLP) ----------
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

# reuse imputed+scaled features from the LR cell (X_tr_sc, X_va_sc)
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(hidden, dropout, lr, epochs=4, batch_size=4096):
    torch.manual_seed(0)
    Xt = torch.tensor(X_tr_sc, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    Xv = torch.tensor(X_va_sc, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val,   dtype=torch.float32).to(device)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    model = MLP(Xt.shape[1], hidden=hidden, dropout=dropout).to(device)
    pos_w = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)], dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            s_val = torch.sigmoid(model(Xv)).cpu().numpy()
        print(f"  epoch {ep+1}: val AUC={roc_auc_score(y_val, s_val):.3f}")
    return model

def score_mlp(model, X_np):
    model.eval()
    with torch.no_grad():
        scores = []
        for i in range(0, len(X_np), 65536):
            xb = torch.tensor(X_np[i:i+65536], dtype=torch.float32, device=device)
            scores.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(scores)

nn_grid = [
    {"hidden": (128, 64), "dropout": 0.2, "lr": 1e-3, "epochs": 4},
    {"hidden": (256, 128, 64), "dropout": 0.3, "lr": 1e-3, "epochs": 4},
]
best_nn, best_nn_val_auc = None, -1
for params in nn_grid:
    print("NN", params)
    model = train_mlp(**params)
    s_tr  = score_mlp(model, X_tr_sc)
    s_val = score_mlp(model, X_va_sc)
    p_tr  = (s_tr  >= 0.5).astype(int)
    p_val = (s_val >= 0.5).astype(int)
    m_tr  = eval_split(y_train, p_tr,  s_tr)
    m_val = eval_split(y_val,   p_val, s_val)
    log_run("MLP", str(params), m_tr, m_val)
    print(f"  -> val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}\n")
    if m_val["roc_auc"] > best_nn_val_auc:
        best_nn, best_nn_val_auc = model, m_val["roc_auc"]
        best_nn_params, best_nn_m_tr, best_nn_m_val = params, m_tr, m_val

print("BEST NN:", best_nn_params, "val AUC =", round(best_nn_val_auc, 3))

using device: cuda
NN {'hidden': (128, 64), 'dropout': 0.2, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.856
  epoch 2: val AUC=0.871
  epoch 3: val AUC=0.871
  epoch 4: val AUC=0.871
  -> val AUC=0.871, val F1=0.156

NN {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.870
  epoch 2: val AUC=0.863
  epoch 3: val AUC=0.873
  epoch 4: val AUC=0.871
  -> val AUC=0.871, val F1=0.157

BEST NN: {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4} val AUC = 0.871


In [17]:
# ---------- Final summary tables ----------
summary_rows = [
    {"model": "Decision Tree",       **fmt(best_dt_m_tr,  best_dt_m_val)},
    {"model": "XGBoost",             **fmt(best_xgb_m_tr, best_xgb_m_val)},
    {"model": "Logistic Regression", **fmt(best_lr_m_tr,  best_lr_m_val)},
    {"model": "Custom NN (MLP)",     **fmt(best_nn_m_tr,  best_nn_m_val)},
]
summary = pd.DataFrame(summary_rows).set_index("model")
print("=== Q7 best-variant summary (train/val) ===")
print(summary)

print("\n=== Q7 full hyperparam grid (every variant tried) ===")
runs_df = pd.DataFrame(all_runs)
# round numeric cols
for c in runs_df.columns:
    if pd.api.types.is_numeric_dtype(runs_df[c]):
        runs_df[c] = runs_df[c].round(3)
print(runs_df.to_string(index=False))

# persist for the report
summary.to_csv("features/q7_summary.csv")
runs_df.to_csv("features/q7_all_runs.csv", index=False)

=== Q7 best-variant summary (train/val) ===
                        accuracy      roc_auc    precision       recall  \
model                                                                     
Decision Tree        0.778/0.721  0.811/0.862  0.145/0.079  0.703/0.832   
XGBoost              0.788/0.731  0.852/0.882  0.157/0.083  0.744/0.855   
Logistic Regression  0.793/0.763  0.821/0.850  0.152/0.087  0.686/0.784   
Custom NN (MLP)      0.788/0.756  0.843/0.871  0.154/0.087  0.718/0.809   

                              f1  
model                             
Decision Tree        0.240/0.144  
XGBoost              0.259/0.152  
Logistic Regression  0.248/0.157  
Custom NN (MLP)      0.253/0.157  

=== Q7 full hyperparam grid (every variant tried) ===
       model                                                              variant  tr_accuracy  tr_roc_auc  tr_precision  tr_recall  tr_f1  val_accuracy  val_roc_auc  val_precision  val_recall  val_f1
DecisionTree                           

### Q7 — Final results table

## Q8–Q9 — Metric definitions and results discussion

## Q10–Q11 — Feature importance: method + top-10 per model

In [18]:
# ---------- Q10/Q11: feature importances ----------
feature_names = list(X_train.columns)

def top_k(importances, k=10):
    idx = np.argsort(importances)[::-1][:k]
    return pd.DataFrame({"feature": [feature_names[i] for i in idx],
                         "importance": importances[idx]})

dt_top  = top_k(best_dt.feature_importances_)
xgb_top = top_k(best_xgb.feature_importances_)
lr_top  = top_k(np.abs(best_lr.coef_[0]))

print("=== DT top-10 ===");  print(dt_top.to_string(index=False))
print("\n=== XGB top-10 ==="); print(xgb_top.to_string(index=False))
print("\n=== LR top-10 (|coef| on standardized features) ==="); print(lr_top.to_string(index=False))

# Save side-by-side for the report
top10 = pd.concat({
    "Decision Tree": dt_top.reset_index(drop=True),
    "XGBoost":       xgb_top.reset_index(drop=True),
    "LogReg":        lr_top.reset_index(drop=True),
}, axis=1)
top10.to_csv("features/q11_top10.csv")

=== DT top-10 ===
                       feature  importance
      posts__PostTypeId__count    0.746240
                posts__Id__sum    0.082045
         users__AboutMe__count    0.046495
             badges__Date__max    0.035193
             comments__Id__sum    0.017891
     posts__OwnerUserId__count    0.014717
      posts__CreationDate__max    0.014611
   comments__CreationDate__max    0.014170
postHistory__CreationDate__max    0.013409
        users__AccountId__mean    0.005199

=== XGB top-10 ===
                       feature  importance
              posts__Id__count    0.442907
        posts__PostTypeId__sum    0.214331
                posts__Id__sum    0.042426
             comments__Id__sum    0.033527
         users__AboutMe__count    0.031571
        users__Location__count    0.015325
          postHistory__Id__sum    0.014939
postHistory__CreationDate__max    0.014772
           badges__Class__mean    0.014297
            badges__Class__sum    0.013245

=== LR top-10 (

### Q11 — Top-10 discussion

## Part 2 — Q12: Graph construction and structural features

In [2]:
import pandas as pd, numpy as np, networkx as nx, time, gc
from collections import defaultdict
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-stack", download=True)
db = dataset.get_db()
task = get_task("rel-stack", "user-engagement", download=True)
train_table = task.get_table("train")
val_table   = task.get_table("val")
test_table  = task.get_table("test")

fm_train = pd.read_pickle("features/fm_train.pkl")
fm_val   = pd.read_pickle("features/fm_val.pkl")
fm_test  = pd.read_pickle("features/fm_test.pkl")

print(fm_train.shape)  # should be (1360850, 99)


/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/abed/.cache/relbench/rel-stack/db...
Done in 6.62 seconds.
(1360850, 99)


In [3]:
# ----- Q12(a): build the FK graph and compute degree -----
import networkx as nx
import time

t0 = time.time()

# 1. Build edge list as a DataFrame (vectorized; no iterrows)
edge_frames = []
for tname, tbl in db.table_dict.items():
    pk = tbl.pkey_col
    for fk_col, ref_table in tbl.fkey_col_to_pkey_table.items():
        sub = tbl.df[[pk, fk_col]].dropna(subset=[fk_col]).copy()
        sub[fk_col] = sub[fk_col].astype("int64")
        sub[pk] = sub[pk].astype("int64")
        e = pd.DataFrame({
            "src": list(zip([tname] * len(sub), sub[pk].tolist())),
            "dst": list(zip([ref_table] * len(sub), sub[fk_col].tolist())),
        })
        edge_frames.append(e)

edges_df = pd.concat(edge_frames, ignore_index=True)
print(f"edges: {len(edges_df):,}   ({time.time()-t0:.1f}s)")

# 2. Build the networkx graph
G = nx.from_pandas_edgelist(edges_df, source="src", target="dst", create_using=nx.Graph())
print(f"graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges   ({time.time()-t0:.1f}s)")

# 3. Degree per user node
users_table = db.table_dict["users"].df
user_ids = users_table["Id"].astype("int64").tolist()
user_nodes = [("users", uid) for uid in user_ids]

deg = dict(G.degree())  # dict: node -> degree
user_degree = pd.Series(
    {uid: deg.get(("users", uid), 0) for uid in user_ids},
    name="graph_degree",
)
user_degree.index.name = "OwnerUserId"
print(f"\nuser degree summary:")
print(user_degree.describe())
print(f"\ntotal time: {time.time()-t0:.1f}s")


edges: 5,812,887   (2.5s)
graph: 4,023,229 nodes, 5,812,887 edges   (21.6s)

user degree summary:
count    255360.000000
mean          9.827741
std         258.061809
min           0.000000
25%           0.000000
50%           1.000000
75%           6.000000
max       70436.000000
Name: graph_degree, dtype: float64

total time: 24.2s


In [4]:
# ----- Q12(b): Weisfeiler-Lehman coloring for K in {3, 5, 10, convergence} -----
import time

def wl_user_colorings(G, user_nodes, snapshot_at=(3, 5, 10), max_iters=15):
    """Run WL hashing on G; keep user-node colors at the requested iterations
    plus the final color when WL converges (or max_iters)."""
    t0 = time.time()
    color = {n: hash(n[0]) for n in G.nodes()}
    n_classes_prev = len(set(color.values()))
    print(f"  iter 0: {n_classes_prev} color classes (= number of tables)")

    adj = G.adj
    user_set = set(user_nodes)
    user_snaps = {}
    converged_at = None
    snapshot_set = set(snapshot_at)

    for it in range(1, max_iters + 1):
        new_color = {}
        for n, _ in color.items():
            nbr = adj[n]
            if nbr:
                nbr_colors = sorted(color[m] for m in nbr)
                new_color[n] = hash((color[n], tuple(nbr_colors)))
            else:
                new_color[n] = hash((color[n], ()))

        if it in snapshot_set:
            user_snaps[it] = {u: new_color[u] for u in user_set if u in new_color}

        n_classes_now = len(set(new_color.values()))
        print(f"  iter {it}: {n_classes_now} color classes   ({time.time()-t0:.1f}s)")

        if n_classes_now == n_classes_prev:
            converged_at = it
            color = new_color
            print(f"  WL converged at iter {it}")
            break

        color = new_color
        n_classes_prev = n_classes_now

    # backfill: if WL converged before some requested K, that K's coloring equals the converged coloring
    final_user_color = {u: color[u] for u in user_set if u in color}
    for k in snapshot_set:
        if k not in user_snaps:
            user_snaps[k] = dict(final_user_color)
    user_snaps["convergence"] = final_user_color
    return user_snaps, converged_at

user_snaps, converged = wl_user_colorings(G, user_nodes, snapshot_at=(3, 5, 10), max_iters=15)
print(f"\nconverged_at: {converged}")
print(f"snapshots: {list(user_snaps.keys())}")
for k in (3, 5, 10, "convergence"):
    print(f"  K={k}: {len(set(user_snaps[k].values())):,} distinct user-node colors")


  iter 0: 7 color classes (= number of tables)
  iter 1: 44514 color classes   (14.4s)
  iter 2: 1092491 color classes   (26.6s)
  iter 3: 1670725 color classes   (39.6s)
  iter 4: 1735101 color classes   (51.4s)
  iter 5: 1744441 color classes   (64.4s)
  iter 6: 1744479 color classes   (77.5s)
  iter 7: 1744479 color classes   (90.1s)
  WL converged at iter 7

converged_at: 7
snapshots: [3, 5, 10, 'convergence']
  K=3: 72,885 distinct user-node colors
  K=5: 84,479 distinct user-node colors
  K=10: 84,480 distinct user-node colors
  K=convergence: 84,480 distinct user-node colors


In [5]:
# ----- Q12(c): graphlet counts via pandas (no need to enumerate subgraphs) -----
import pandas as pd
import time

t0 = time.time()

# Graphlet 1: user-comment-post-user (user commented on their own post)
comments = db.table_dict["comments"].df[["UserId", "PostId"]].dropna(subset=["UserId", "PostId"]).copy()
comments["UserId"] = comments["UserId"].astype("int64")
comments["PostId"] = comments["PostId"].astype("int64")

posts_owner = db.table_dict["posts"].df[["Id", "OwnerUserId"]].dropna(subset=["OwnerUserId"]).copy()
posts_owner["OwnerUserId"] = posts_owner["OwnerUserId"].astype("int64")
posts_owner = posts_owner.rename(columns={"Id": "PostId", "OwnerUserId": "PostOwnerId"})

m = comments.merge(posts_owner, on="PostId", how="inner")
self_comments = m[m["UserId"] == m["PostOwnerId"]]
gl1_count = self_comments.groupby("UserId").size().rename("self_comment_count")
print(f"graphlet 1 (self-comment): {len(self_comments):,} matching tuples; {gl1_count.shape[0]:,} users involved")

# Graphlet 2: user-post-postLinks-post-user (user has two linked posts)
links = db.table_dict["postLinks"].df[["PostId", "RelatedPostId"]].dropna().copy()
links["PostId"] = links["PostId"].astype("int64")
links["RelatedPostId"] = links["RelatedPostId"].astype("int64")

A = posts_owner.rename(columns={"PostId": "PostId", "PostOwnerId": "OwnerA"})
B = posts_owner.rename(columns={"PostId": "RelatedPostId", "PostOwnerId": "OwnerB"})

m2 = links.merge(A, on="PostId", how="inner").merge(B, on="RelatedPostId", how="inner")
linked_same = m2[m2["OwnerA"] == m2["OwnerB"]]
gl2_count = linked_same.groupby("OwnerA").size().rename("linked_own_posts_count")
gl2_count.index.name = "UserId"
print(f"graphlet 2 (linked own posts): {len(linked_same):,} matching tuples; {gl2_count.shape[0]:,} users involved")
print(f"\ntotal time: {time.time()-t0:.1f}s")


graphlet 1 (self-comment): 202,465 matching tuples; 37,631 users involved
graphlet 2 (linked own posts): 3,591 matching tuples; 1,888 users involved

total time: 0.2s


In [6]:
# ----- Q12: assemble graph features and add to feature_matrix -----
import time, os

t0 = time.time()

def add_graph_features(fm):
    out = fm.copy()
    # join on OwnerUserId
    out = out.merge(user_degree, left_on="OwnerUserId", right_index=True, how="left")
    out["graph_degree"] = out["graph_degree"].fillna(0).astype("int64")

    # WL colorings: convert color hashes -> stable integer IDs (so XGB/DT/LR can use them as categorical)
    for k in (3, 5, 10, "convergence"):
        col_name = f"wl_K{k}"
        cmap = user_snaps[k]  # dict: (table,id) -> color_hash
        # build a Series mapping uid -> color
        uid_to_color = {uid: cmap.get(("users", uid)) for uid in users_table["Id"].astype("int64")}
        # factorize colors to integer IDs (per the assignment - "color id")
        s = pd.Series(uid_to_color, name=col_name)
        codes, _ = pd.factorize(s, use_na_sentinel=True)
        s_codes = pd.Series(codes, index=s.index, name=col_name)
        out = out.merge(s_codes, left_on="OwnerUserId", right_index=True, how="left")
        out[col_name] = out[col_name].fillna(-1).astype("int64")

    # graphlet counts
    out = out.merge(gl1_count.rename("self_comment_count"),
                    left_on="OwnerUserId", right_index=True, how="left")
    out["self_comment_count"] = out["self_comment_count"].fillna(0).astype("int64")

    out = out.merge(gl2_count.rename("linked_own_posts_count"),
                    left_on="OwnerUserId", right_index=True, how="left")
    out["linked_own_posts_count"] = out["linked_own_posts_count"].fillna(0).astype("int64")
    return out

fm_train_g = add_graph_features(fm_train)
fm_val_g   = add_graph_features(fm_val)
fm_test_g  = add_graph_features(fm_test)

os.makedirs("features", exist_ok=True)
fm_train_g.to_pickle("features/fm_train_graph.pkl")
fm_val_g.to_pickle("features/fm_val_graph.pkl")
fm_test_g.to_pickle("features/fm_test_graph.pkl")

new_cols = [c for c in fm_train_g.columns if c not in fm_train.columns]
print(f"new graph feature columns ({len(new_cols)}):\n  " + "\n  ".join(new_cols))
print(f"\nfm_train_g.shape: {fm_train_g.shape}   ({time.time()-t0:.1f}s)")
print(f"fm_val_g.shape:   {fm_val_g.shape}")
print(f"fm_test_g.shape:  {fm_test_g.shape}")


new graph feature columns (7):
  graph_degree
  wl_K3
  wl_K5
  wl_K10
  wl_Kconvergence
  self_comment_count
  linked_own_posts_count

fm_train_g.shape: (1360850, 106)   (15.1s)
fm_val_g.shape:   (85838, 106)
fm_test_g.shape:  (88137, 105)


## Q13 — Retraining with graph features

In [1]:
# ----- Q13 recovery cell -----
# Run this AFTER restarting the kernel (Kernel -> Restart) to recover from the Part-2 OOM.
# It re-imports everything we need and reloads the saved feature matrices, without
# pulling the 4M-node networkx graph or the WL color dicts back into memory.

import os, gc
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ----- helpers (re-defined after kernel restart) -----
def prepare_xy(fm, drop_cols=("OwnerUserId", "timestamp", "contribution")):
    y = fm["contribution"].values.astype("int64") if "contribution" in fm.columns else None
    X = fm.drop(columns=[c for c in drop_cols if c in fm.columns]).copy()
    for c in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[c]):
            X[c] = X[c].astype("int64")
            X[c] = X[c].mask(X[c] < 0, -1)
        elif X[c].dtype == "object":
            X[c] = pd.to_numeric(X[c], errors="coerce")
        else:
            X[c] = X[c].astype("float64")
    return X, y

def eval_split(y_true, y_pred, y_score):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "roc_auc":   roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def fmt(metrics_tr, metrics_val):
    return {k: f"{metrics_tr[k]:.3f}/{metrics_val[k]:.3f}" for k in metrics_tr}

all_runs_g = []
def log_run_g(model_name, variant, m_tr, m_val):
    all_runs_g.append({"model": model_name, "variant": variant,
                       **{f"tr_{k}": v for k, v in m_tr.items()},
                       **{f"val_{k}": v for k, v in m_val.items()}})

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

# ----- load pickled enriched feature matrices -----
fm_train_g = pd.read_pickle("features/fm_train_graph.pkl")
fm_val_g   = pd.read_pickle("features/fm_val_graph.pkl")

X_train_g, y_train_g = prepare_xy(fm_train_g)
X_val_g,   y_val_g   = prepare_xy(fm_val_g)

# Drop fm_*_g once we have X/y to save RAM
del fm_train_g, fm_val_g; gc.collect()

# Load Part 1 summary (for the +graph vs tabular-only comparison)
part1 = pd.read_csv("features/q7_summary.csv", index_col=0)
print("Part 1 val AUCs (from features/q7_summary.csv):")
for m in part1.index:
    print(f"  {m}: {part1.loc[m, 'roc_auc'].split('/')[1]}")

# Hyperparameter grids (same as Part 1)
dt_grid  = [{"max_depth": 5}, {"max_depth": 10}, {"max_depth": 20},
            {"max_depth": None, "min_samples_leaf": 50}]
xgb_grid = [{"max_depth": 4,  "n_estimators": 200, "learning_rate": 0.10},
            {"max_depth": 6,  "n_estimators": 200, "learning_rate": 0.10},
            {"max_depth": 8,  "n_estimators": 400, "learning_rate": 0.05}]
lr_grid  = [{"C": 0.1}, {"C": 1.0}, {"C": 10.0}]
nn_grid  = [{"hidden": (128, 64),      "dropout": 0.2, "lr": 1e-3, "epochs": 4},
            {"hidden": (256, 128, 64), "dropout": 0.3, "lr": 1e-3, "epochs": 4}]

print(f"\nX_train_g: {X_train_g.shape}, X_val_g: {X_val_g.shape}")
print(f"y_train_g pos-rate: {y_train_g.mean():.4f}")


device: cuda
Part 1 val AUCs (from features/q7_summary.csv):
  Decision Tree: 0.862
  XGBoost: 0.882
  Logistic Regression: 0.850
  Custom NN (MLP): 0.871

X_train_g: (1360850, 103), X_val_g: (85838, 103)
y_train_g pos-rate: 0.0500


In [6]:
# ----- Q13: Decision Tree -----
import gc
X_train_dt_g = X_train_g.fillna(-1).values.astype(np.float32)   # float32 halves memory
X_val_dt_g   = X_val_g.fillna(-1).values.astype(np.float32)

best_dt_g, best_dt_g_val_auc = None, -1
for params in dt_grid:
    clf = DecisionTreeClassifier(class_weight="balanced", random_state=0, **params)
    clf.fit(X_train_dt_g, y_train_g)
    s_tr  = clf.predict_proba(X_train_dt_g)[:, 1]
    s_val = clf.predict_proba(X_val_dt_g)[:, 1]
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("DecisionTree", str(params), m_tr, m_val)
    print(f"DT {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_dt_g_val_auc:
        best_dt_g, best_dt_g_val_auc = clf, m_val["roc_auc"]
        best_dt_g_params, best_dt_g_m_tr, best_dt_g_m_val = params, m_tr, m_val

print(f"\nBEST DT_g: {best_dt_g_params}  val AUC={best_dt_g_val_auc:.3f}")
del X_train_dt_g, X_val_dt_g; gc.collect()


DT {'max_depth': 5}: val AUC=0.921, val F1=0.253
DT {'max_depth': 10}: val AUC=0.933, val F1=0.378
DT {'max_depth': 20}: val AUC=0.697, val F1=0.440
DT {'max_depth': None, 'min_samples_leaf': 50}: val AUC=0.923, val F1=0.400

BEST DT_g: {'max_depth': 10}  val AUC=0.933


214

In [2]:
# ----- Q13: XGBoost -----
import gc
pos_ratio_g = float((y_train_g == 0).sum() / (y_train_g == 1).sum())
X_train_xgb = X_train_g.values.astype(np.float32)
X_val_xgb   = X_val_g.values.astype(np.float32)

best_xgb_g, best_xgb_g_val_auc = None, -1
for params in xgb_grid:
    clf = XGBClassifier(scale_pos_weight=pos_ratio_g, tree_method="hist",
                        n_jobs=-1, random_state=0, eval_metric="logloss", **params)
    clf.fit(X_train_xgb, y_train_g)
    s_tr  = clf.predict_proba(X_train_xgb)[:, 1]
    s_val = clf.predict_proba(X_val_xgb)[:, 1]
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("XGBoost", str(params), m_tr, m_val)
    print(f"XGB {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_xgb_g_val_auc:
        best_xgb_g, best_xgb_g_val_auc = clf, m_val["roc_auc"]
        best_xgb_g_params, best_xgb_g_m_tr, best_xgb_g_m_val = params, m_tr, m_val

print(f"\nBEST XGB_g: {best_xgb_g_params}  val AUC={best_xgb_g_val_auc:.3f}")
del X_train_xgb, X_val_xgb; gc.collect()


XGB {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.977, val F1=0.444
XGB {'max_depth': 6, 'n_estimators': 200, 'learning_rate': 0.1}: val AUC=0.983, val F1=0.533
XGB {'max_depth': 8, 'n_estimators': 400, 'learning_rate': 0.05}: val AUC=0.984, val F1=0.618

BEST XGB_g: {'max_depth': 8, 'n_estimators': 400, 'learning_rate': 0.05}  val AUC=0.984


0

In [3]:
# ----- Q13: Logistic Regression -----
import gc
imp_g    = SimpleImputer(strategy="median").fit(X_train_g.values)
X_tr_imp_g = imp_g.transform(X_train_g.values).astype(np.float32)
X_va_imp_g = imp_g.transform(X_val_g.values).astype(np.float32)
scaler_g = StandardScaler().fit(X_tr_imp_g)
X_tr_sc_g = scaler_g.transform(X_tr_imp_g).astype(np.float32)
X_va_sc_g = scaler_g.transform(X_va_imp_g).astype(np.float32)
del X_tr_imp_g, X_va_imp_g; gc.collect()

best_lr_g, best_lr_g_val_auc = None, -1
for params in lr_grid:
    clf = LogisticRegression(class_weight="balanced", max_iter=2000, solver="lbfgs",
                             n_jobs=-1, random_state=0, **params)
    clf.fit(X_tr_sc_g, y_train_g)
    s_tr  = clf.predict_proba(X_tr_sc_g)[:, 1]
    s_val = clf.predict_proba(X_va_sc_g)[:, 1]
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("LogReg", str(params), m_tr, m_val)
    print(f"LR {params}: val AUC={m_val['roc_auc']:.3f}, val F1={m_val['f1']:.3f}")
    if m_val["roc_auc"] > best_lr_g_val_auc:
        best_lr_g, best_lr_g_val_auc = clf, m_val["roc_auc"]
        best_lr_g_params, best_lr_g_m_tr, best_lr_g_m_val = params, m_tr, m_val

print(f"\nBEST LR_g: {best_lr_g_params}  val AUC={best_lr_g_val_auc:.3f}")
# keep X_tr_sc_g, X_va_sc_g around -> the NN cell reuses them


/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 8 10]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/home/abed/miniconda3/envs/structml1/lib/python3.11/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [ 8 10]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


LR {'C': 0.1}: val AUC=0.919, val F1=0.258
LR {'C': 1.0}: val AUC=0.927, val F1=0.288
LR {'C': 10.0}: val AUC=0.928, val F1=0.290

BEST LR_g: {'C': 10.0}  val AUC=0.928


In [4]:
# ----- Q13: Neural Net -----
import gc

def train_mlp_g(hidden, dropout, lr, epochs=4, batch_size=4096):
    torch.manual_seed(0)
    Xt = torch.tensor(X_tr_sc_g, dtype=torch.float32)
    yt = torch.tensor(y_train_g, dtype=torch.float32)
    Xv = torch.tensor(X_va_sc_g, dtype=torch.float32).to(device)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)
    model = MLP(Xt.shape[1], hidden=hidden, dropout=dropout).to(device)
    pos_w = torch.tensor([(y_train_g == 0).sum() / max((y_train_g == 1).sum(), 1)],
                          dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            s_val = torch.sigmoid(model(Xv)).cpu().numpy()
        print(f"  epoch {ep+1}: val AUC={roc_auc_score(y_val_g, s_val):.3f}")
    return model

def score_mlp_g(model, X_np):
    model.eval()
    with torch.no_grad():
        out = []
        for i in range(0, len(X_np), 65536):
            xb = torch.tensor(X_np[i:i+65536], dtype=torch.float32, device=device)
            out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)

best_nn_g, best_nn_g_val_auc = None, -1
for params in nn_grid:
    print("NN", params)
    model = train_mlp_g(**params)
    s_tr  = score_mlp_g(model, X_tr_sc_g)
    s_val = score_mlp_g(model, X_va_sc_g)
    m_tr  = eval_split(y_train_g, (s_tr  >= 0.5).astype(int), s_tr)
    m_val = eval_split(y_val_g,   (s_val >= 0.5).astype(int), s_val)
    log_run_g("MLP", str(params), m_tr, m_val)
    print(f"  -> val AUC={m_val['roc_auc']:.3f}\n")
    if m_val["roc_auc"] > best_nn_g_val_auc:
        best_nn_g, best_nn_g_val_auc = model, m_val["roc_auc"]
        best_nn_g_params, best_nn_g_m_tr, best_nn_g_m_val = params, m_tr, m_val

print(f"BEST NN_g: {best_nn_g_params}  val AUC={best_nn_g_val_auc:.3f}")
del X_tr_sc_g, X_va_sc_g; gc.collect()


NN {'hidden': (128, 64), 'dropout': 0.2, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.893
  epoch 2: val AUC=0.903
  epoch 3: val AUC=0.896
  epoch 4: val AUC=0.929
  -> val AUC=0.929

NN {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}
  epoch 1: val AUC=0.893
  epoch 2: val AUC=0.905
  epoch 3: val AUC=0.922
  epoch 4: val AUC=0.938
  -> val AUC=0.938

BEST NN_g: {'hidden': (256, 128, 64), 'dropout': 0.3, 'lr': 0.001, 'epochs': 4}  val AUC=0.938


0

In [7]:
# ----- Q13 results summary + +graph vs tabular-only comparison -----
summary_rows_g = [
    {"model": "Decision Tree",       **fmt(best_dt_g_m_tr,  best_dt_g_m_val)},
    {"model": "XGBoost",             **fmt(best_xgb_g_m_tr, best_xgb_g_m_val)},
    {"model": "Logistic Regression", **fmt(best_lr_g_m_tr,  best_lr_g_m_val)},
    {"model": "Custom NN (MLP)",     **fmt(best_nn_g_m_tr,  best_nn_g_m_val)},
]
summary_g = pd.DataFrame(summary_rows_g).set_index("model")
print("=== Q13 best-variant summary WITH GRAPH FEATURES (train/val) ===")
print(summary_g)

# Side-by-side with Part 1 (from disk)
part1 = pd.read_csv("features/q7_summary.csv", index_col=0)
def _val_auc(s): return float(s.split("/")[1])

p1_aucs = {m: _val_auc(part1.loc[m, "roc_auc"]) for m in part1.index}
p2_aucs = {
    "Decision Tree":       best_dt_g_m_val["roc_auc"],
    "XGBoost":             best_xgb_g_m_val["roc_auc"],
    "Logistic Regression": best_lr_g_m_val["roc_auc"],
    "Custom NN (MLP)":     best_nn_g_m_val["roc_auc"],
}
cmp = pd.DataFrame({"tabular_only": p1_aucs, "+graph": p2_aucs})
cmp["delta"] = cmp["+graph"] - cmp["tabular_only"]
print("\n=== Val ROC-AUC: tabular-only vs +graph ===")
print(cmp.round(4))

summary_g.to_csv("features/q13_summary.csv")
pd.DataFrame(all_runs_g).to_csv("features/q13_all_runs.csv", index=False)


=== Q13 best-variant summary WITH GRAPH FEATURES (train/val) ===
                        accuracy      roc_auc    precision       recall  \
model                                                                     
Decision Tree        0.866/0.922  0.955/0.933  0.263/0.243  0.941/0.846   
XGBoost              0.899/0.971  0.972/0.984  0.327/0.486  0.965/0.849   
Logistic Regression  0.897/0.890  0.929/0.928  0.298/0.177  0.788/0.800   
Custom NN (MLP)      0.853/0.879  0.939/0.938  0.239/0.169  0.885/0.843   

                              f1  
model                             
Decision Tree        0.412/0.378  
XGBoost              0.488/0.618  
Logistic Regression  0.432/0.290  
Custom NN (MLP)      0.376/0.282  

=== Val ROC-AUC: tabular-only vs +graph ===
                     tabular_only  +graph   delta
Decision Tree               0.862  0.9332  0.0712
XGBoost                     0.882  0.9838  0.1018
Logistic Regression         0.850  0.9275  0.0775
Custom NN (MLP)             0

## Q14 — Top-10 with graph features

In [8]:
# ----- Q14: top-10 features after adding graph features -----
feature_names_g = list(X_train_g.columns)
def top_k_g(importances, k=10):
    idx = np.argsort(importances)[::-1][:k]
    return pd.DataFrame({"feature": [feature_names_g[i] for i in idx],
                         "importance": importances[idx]})

dt_top_g  = top_k_g(best_dt_g.feature_importances_)
xgb_top_g = top_k_g(best_xgb_g.feature_importances_)
lr_top_g  = top_k_g(np.abs(best_lr_g.coef_[0]))

print("=== DT top-10 (with graph features) ===")
print(dt_top_g.to_string(index=False))
print("\n=== XGB top-10 (with graph features) ===")
print(xgb_top_g.to_string(index=False))
print("\n=== LR top-10 (with graph features) ===")
print(lr_top_g.to_string(index=False))

# Highlight which graph features made the top 10s
graph_cols = {"graph_degree", "wl_K3", "wl_K5", "wl_K10", "wl_Kconvergence",
              "self_comment_count", "linked_own_posts_count"}
print("\n=== Graph features that landed in any top-10 ===")
in_dt  = set(dt_top_g["feature"])  & graph_cols
in_xgb = set(xgb_top_g["feature"]) & graph_cols
in_lr  = set(lr_top_g["feature"])  & graph_cols
print(f"  Decision Tree:  {sorted(in_dt)}")
print(f"  XGBoost:        {sorted(in_xgb)}")
print(f"  LogReg:         {sorted(in_lr)}")

pd.concat({"DT": dt_top_g.reset_index(drop=True),
           "XGB": xgb_top_g.reset_index(drop=True),
           "LR":  lr_top_g.reset_index(drop=True)}, axis=1
).to_csv("features/q14_top10.csv")


=== DT top-10 (with graph features) ===
                         feature  importance
                    graph_degree    0.743003
          postHistory__Id__count    0.034598
postHistory__RevisionGUID__count    0.030617
               badges__Id__count    0.016402
      postHistory__PostId__count    0.014929
         comments__PostId__count    0.012651
           badges__UserId__count    0.011914
           comments__Text__count    0.009510
 comments__ContentLicense__count    0.009387
        postHistory__Text__count    0.008059

=== XGB top-10 (with graph features) ===
                           feature  importance
                      graph_degree    0.248128
            postHistory__Id__count    0.114622
                 badges__Id__count    0.054744
postHistory__ContentLicense__count    0.039472
               comments__Id__count    0.033720
          postHistory__Text__count    0.031334
                comments__Id__mean    0.027561
                   posts__Id__mean    0.022127
